# KoGPT2 파인튜닝 (파라메트릭 학습) — Colab GPU

## 파라메트릭 vs 논파라메트릭 학습
- **파라메트릭(parametric) 학습** = 지식을 **모델 가중치 안**에 저장. 이 노트북이 하는 일(파인튜닝).
- **논파라메트릭(non-parametric) 학습** = 지식을 **외부 벡터DB(ChromaDB)**에 저장하고 검색(RAG). `rag_chroma/` 가 담당.

이 노트북은 **HuggingFace 공개 한국어 instruction 데이터셋(KoAlpaca)** 으로 **KoGPT2(skt/kogpt2-base-v2)** 를 파인튜닝한다.

> Colab 상단 메뉴 → **런타임 → 런타임 유형 변경 → GPU(T4)** 로 설정하고 실행하세요.

In [ ]:
# 1) 의존성 설치
#    ⚠️ KoGPT2 토크나이저가 최신 transformers/tokenizers 에서 한국어를 못 읽고 깨지는 문제가 있어
#       한국어 라운드트립이 검증된 버전으로 고정한다.
!pip -q install "transformers==4.57.6" "tokenizers==0.22.2" datasets accelerate huggingface_hub
print("설치 완료 — 만약 4번 셀 토크나이저 검사에서 실패하면 [런타임 → 세션 다시 시작] 후 다시 [모두 실행] 하세요.")

In [ ]:
# 2) GPU 확인
import torch
print('GPU 사용 가능:', torch.cuda.is_available())
print('장치:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

In [ ]:
# 3) HuggingFace 공개 한국어 데이터셋 로드 (KoAlpaca - instruction/output)
from datasets import load_dataset

dataset = load_dataset('beomi/KoAlpaca-v1.1a', split='train')
print(dataset)
print('예시:', dataset[0])

# 빠른 실습을 위해 일부만 사용 (전체로 하려면 이 줄을 주석 처리)
dataset = dataset.select(range(3000))
print('사용할 샘플 수:', len(dataset))

In [ ]:
# 4) KoGPT2 모델 + 토크나이저 로드
from transformers import AutoTokenizer, AutoModelForCausalLM

MODEL_NAME = 'skt/kogpt2-base-v2'
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_NAME,
    bos_token='</s>', eos_token='</s>', unk_token='<unk>',
    pad_token='<pad>', mask_token='<mask>',
)
model = AutoModelForCausalLM.from_pretrained(MODEL_NAME)
print('파라미터 수:', model.num_parameters())

# ⚠️ 토크나이저 정상 작동 검증 (한국어 라운드트립).
#    여기서 깨지면 학습해도 헛수고 → 즉시 멈춰서 버전 문제를 먼저 잡는다.
_test = '파이썬이 뭐야?'
_round = tokenizer.decode(tokenizer.encode(_test), skip_special_tokens=True)
print('토크나이저 한국어 테스트:', repr(_round))
assert _test.replace(' ', '') in _round.replace(' ', ''), (
    '❌ 토크나이저가 한국어를 못 읽습니다(바이트 폴백). '
    '1번 셀로 버전을 고정한 뒤 [런타임 → 세션 다시 시작 및 모두 실행] 하세요.'
)
print('✅ 토크나이저 정상 — 학습 진행 가능')

In [ ]:
# 5) 데이터 포맷팅 + 토크나이즈
#    '질문-답변' 형식으로 만들어, 질문이 들어오면 답변을 생성하도록 학습한다.
PROMPT = '### 질문: {q}\n### 답변: {a}'
MAX_LEN = 256

def format_example(ex):
    text = PROMPT.format(q=ex['instruction'].strip(), a=str(ex['output']).strip())
    text = text + tokenizer.eos_token
    # 패딩 없이 토큰화 → 패딩/라벨은 DataCollator 가 자동 처리(패드는 -100 으로 마스킹)
    return tokenizer(text, truncation=True, max_length=MAX_LEN)

tokenized = dataset.map(format_example, remove_columns=dataset.column_names)
print(tokenized)

In [ ]:
# 6) 파인튜닝 (파라메트릭 학습 — 모델 가중치를 갱신)
from transformers import Trainer, TrainingArguments, DataCollatorForLanguageModeling

# mlm=False → 인과적 언어모델용. labels 자동 생성 + <pad> 를 -100 으로 마스킹.
collator = DataCollatorForLanguageModeling(tokenizer, mlm=False)

args = TrainingArguments(
    output_dir='kogpt2-finetuned',
    per_device_train_batch_size=8,
    num_train_epochs=3,
    learning_rate=3e-5,
    fp16=False,            # ⚠️ KoGPT2(GPT-2)는 fp16 에서 발산(NaN)해 출력이 깨질 수 있음 → fp32 사용
    warmup_ratio=0.1,      # 초반 학습 안정화
    max_grad_norm=1.0,     # 그래디언트 클리핑(발산 방지)
    logging_steps=50,
    save_strategy='epoch',
    report_to='none',
)

trainer = Trainer(model=model, args=args, train_dataset=tokenized, data_collator=collator)
trainer.train()

# 학습이 정상이면 loss 가 점점 줄어들어야 함 (예: 3.x → 2.x). 계속 0 이나 nan 이면 문제.

In [ ]:
# 7) 학습된 모델 저장
save_dir = 'kogpt2-finetuned'
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print('저장 완료:', save_dir)

In [ ]:
# 8) 생성 테스트 — 파인튜닝 전/후 비교
import torch

def ask(question, max_new_tokens=80):
    prompt = '### 질문: ' + question + '\n### 답변:'
    ids = tokenizer.encode(prompt, return_tensors='pt').to(model.device)
    out = model.generate(ids, max_new_tokens=max_new_tokens, do_sample=True,
                         top_p=0.92, temperature=0.7, repetition_penalty=1.3,
                         no_repeat_ngram_size=3, pad_token_id=tokenizer.pad_token_id,
                         eos_token_id=tokenizer.eos_token_id)
    text = tokenizer.decode(out[0], skip_special_tokens=True)
    return text[len(prompt):].split('### 질문')[0].strip()

for q in ['파이썬이 뭐야?', '건강하게 사는 방법 알려줘', '딥러닝과 머신러닝의 차이는?']:
    print('Q:', q)
    print('A:', ask(q))
    print()

## 9) 학습된 모델 가져오기 (둘 중 하나)

**방법 A — HuggingFace Hub 에 올리기 (권장)**: 다른 곳(우리 챗봇)에서 모델 이름으로 바로 불러올 수 있다.

In [ ]:
# 방법 A: HF Hub 업로드 (HF 계정/토큰 필요)
from huggingface_hub import notebook_login
notebook_login()  # 토큰 입력

REPO = 'your-username/kogpt2-koalpaca'   # 본인 계정으로 바꾸세요
model.push_to_hub(REPO)
tokenizer.push_to_hub(REPO)
print('업로드 완료:', REPO)
print('우리 챗봇에서:  export FINETUNED_MODEL_PATH=' + REPO)

In [ ]:
# 방법 B: zip 으로 다운로드해서 로컬 챗봇에 넣기
!zip -r kogpt2-finetuned.zip kogpt2-finetuned
from google.colab import files
files.download('kogpt2-finetuned.zip')
# 내려받은 폴더를 rag_chroma/ 안에 풀고:  export FINETUNED_MODEL_PATH=./kogpt2-finetuned